<a href="https://colab.research.google.com/github/SinnottKayleigh/B2B-Sales-Algos/blob/main/Actor_Critic_Model_(RL).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The actor-critic model can help to optimise the sales process by learning which actions lead to successful deals, in varying customer scenarios.


**Actor-Critic Model:**

**Actor: **Makes decisions by selecting actions based on the current policy. Its responsibility lies in exploring the action space to maximise expected cumulative rewards. By continuously refining the policy, the actor adapts to the dynamic nature of the environment.

**Critic:** Evaluates the actions taken by the actor; estimating the value of quality of these actions by providing feedback on their performance. The critic guides the actor towards actions that lead to higher expected returns, contributing to the overall improvement of the learning process.

**Policy (Actor):** Denoted as π(a∣s), represents the probabulity of taking action 'a' in state 's'. The policy is modelled by the actor network and its parameters are denoted by 'θ'.

**Value Function (Critic): **The value function, denoted by V(s), estimates the expected cumulative reward starting from state 's'. Modelled by the critic network and its parameters are denoted by 'w'.

**State Representation:**
- Customer profile features (company size, industry, budget)
- Interaction history (emails opened, calls answered)
- Current stage in sales pipeline
- Time since last contact
- Previous responses to outreach

**Actions: **
- Recommend specific email templates
- Suggest optimal contact timing
- Determine personalization level
- Recommend call scripts or talking points
- Prioritize prospects for follow-up

**Reward Function:**
- Prospect response rates
- Progression through sales stages
- Meeting scheduled (small positive reward)
- Deal closure (large positive reward)
- Deal value (scaled reward)
- Customer churn (negative reward)

**Use Cases for Prospectoro:**
- Resource Allocation
- Continuous Improvement; learns and improves based on collected data.
- Adaptable to changing scenarios and market conditions.
- Timing Optimisation; learns optimal timing for follow-ups based on prospect behavior patterns.

Implementation Considerations:
- Data Requirements; the need for a significant amount of historical data.
- Feature Engineering; the state representation must be carefully designed to capture relevant aspects of customers and interactions.
- Exploration - Explotation Balance; Configure the system to explore different actions before settling on optimal strategies.
- Integration; APIs to connect to the RL system, coupled with the CRM database for decision making in real time.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

class ActorCriticCRM:
    def __init__(self, state_dim, action_dim, learning_rate=0.001, gamma=0.99):
        """
        Initialize Actor-Critic model for CRM optimization

        Args:
            state_dim: Dimension of the state representation (customer/prospect features)
            action_dim: Dimension of the action space (different sales actions)
            learning_rate: Learning rate for both actor and critic networks
            gamma: Discount factor for future rewards
        """
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma

        # Actor and Critic networks
        self.actor = self._build_actor_network(learning_rate)
        self.critic = self._build_critic_network(learning_rate)

        # For storing training data
        self.states = []
        self.actions = []
        self.rewards = []

    def _build_actor_network(self, learning_rate):
        """Build the actor network that predicts the best sales action"""
        inputs = Input(shape=(self.state_dim,))
        x = Dense(64, activation='relu')(inputs)
        x = Dense(32, activation='relu')(x)
        outputs = Dense(self.action_dim, activation='softmax')(x)

        model = Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer=Adam(learning_rate=learning_rate))
        return model

    def _build_critic_network(self, learning_rate):
        """Build the critic network that evaluates the value of states"""
        inputs = Input(shape=(self.state_dim,))
        x = Dense(64, activation='relu')(inputs)
        x = Dense(32, activation='relu')(x)
        outputs = Dense(1, activation='linear')(x)

        model = Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
        return model

    def get_action(self, state):
        """
        Get recommended sales action for a given prospect state

        Args:
            state: Current state of the prospect/customer interaction

        Returns:
            action_index: Index of the recommended action
            action_probs: Probability distribution over all actions
        """
        state = np.reshape(state, [1, self.state_dim])
        action_probs = self.actor.predict(state, verbose=0)[0]
        action_index = np.random.choice(self.action_dim, p=action_probs)
        return action_index, action_probs

    def get_critic_value(self, state):
        """Get the critic's estimated value of the current state"""
        state = np.reshape(state, [1, self.state_dim])
        return self.critic.predict(state, verbose=0)[0][0]

    def store_transition(self, state, action, reward):
        """Store the experience for training"""
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)

    def train(self):
        """Train the Actor-Critic model on collected sales interaction data"""
        if len(self.states) == 0:
            return

        # Convert to numpy arrays
        states = np.vstack(self.states)
        actions = np.array(self.actions)
        rewards = np.array(self.rewards)

        # Calculate advantages and returns
        values = self.critic.predict(states, verbose=0).flatten()
        advantages = rewards - values

        # Train critic
        self.critic.fit(states, rewards, verbose=0, epochs=1)

        # Create one-hot actions
        action_onehot = np.zeros((len(actions), self.action_dim))
        action_onehot[np.arange(len(actions)), actions] = 1

        # Custom loss function for the actor
        with tf.GradientTape() as tape:
            probs = self.actor(states, training=True)
            selected_action_probs = tf.reduce_sum(probs * action_onehot, axis=1)
            # Policy gradient loss with advantages
            loss = -tf.reduce_sum(tf.math.log(selected_action_probs) * advantages)

        # Apply gradients to the actor
        grads = tape.gradient(loss, self.actor.trainable_variables)
        self.actor.optimizer.apply_gradients(zip(grads, self.actor.trainable_variables))

        # Clear memory
        self.states = []
        self.actions = []
        self.rewards = []

    def save_models(self, actor_path, critic_path):
        """Save the trained models"""
        self.actor.save(actor_path)
        self.critic.save(critic_path)

    def load_models(self, actor_path, critic_path):
        """Load trained models"""
        self.actor = tf.keras.models.load_model(actor_path)
        self.critic = tf.keras.models.load_model(critic_path)

# Example implementation for Prospectoro CRM
def integrate_with_prospectoro():
    """
    Example of how to integrate Actor-Critic with Prospectoro CRM
    """
    # Define state and action dimensions based on your CRM data
    state_dim = 20  # Example: 20 features describing prospect and history
    action_dim = 5  # Example: 5 different sales actions

    # Initialize the Actor-Critic model
    ac_model = ActorCriticCRM(state_dim, action_dim)

    # Example training loop (would connect to your actual CRM data)
    for episode in range(100):
        # Get batch of prospects from CRM
        prospects = get_prospects_from_crm()  # This would be your CRM API call

        for prospect in prospects:
            # Get current state representation of the prospect
            state = extract_prospect_features(prospect)  # Your feature extraction function

            # Get recommended action from the model
            action_index, _ = ac_model.get_action(state)

            # Execute action in CRM (e.g., send email, schedule call)
            execute_sales_action(prospect, action_index)  # Your CRM API call

            # Wait for outcome and get reward (e.g., did prospect respond?)
            reward = get_action_outcome(prospect, action_index)  # Your reward calculation

            # Store this experience
            ac_model.store_transition(state, action_index, reward)

        # Train the model after collecting batch of experiences
        ac_model.train()

        # Periodically save the model
        if episode % 10 == 0:
            ac_model.save_models("prospectoro_actor.h5", "prospectoro_critic.h5")

# Example helper functions (these would connect to your actual CRM)
def get_prospects_from_crm():
    """Get batch of prospects from Prospectoro CRM"""
    # This would be your CRM API call
    return []  # Return list of prospect objects

def extract_prospect_features(prospect):
    """Extract relevant features from prospect data"""
    # Convert prospect data to numerical feature vector
    features = np.zeros(20)  # Example with 20 features
    # Fill features based on prospect data
    return features

def execute_sales_action(prospect, action_index):
    """Execute the recommended sales action in the CRM"""
    # Map action_index to actual CRM operations
    actions = {
        0: "send_email_template_1",
        1: "send_email_template_2",
        2: "schedule_call",
        3: "send_proposal",
        4: "offer_discount"
    }
    action = actions[action_index]
    # Execute in CRM via API

def get_action_outcome(prospect, action_index):
    """Calculate reward based on prospect's response to the action"""
    # This would check CRM for outcomes and calculate reward
    # Example: +1 for email open, +5 for response, +20 for meeting, +100 for deal
    return 0  # Placeholder

**State Representation in B2B Sales Context:**

**Prospect/Company Demographics (Static Features)**

**Firmographic Attributes**

- Company size (employee count)
- Annual revenue
- Industry sector
- Geographic location
- Company age
- Funding status
- Market position/ranking

**Technographic Data**

- Current technology stack
- Recent technology investments
- Digital maturity level
- Software/tools currently used

**Interaction History (Dynamic Features)**

**Communication Metrics**

- Number of previous interactions
- Time since last contact
- Average response time
- Communication channels used
- Email open rates
- Click-through rates on sent materials
- Response sentiment analysis scores

***Engagement Scoring***

- Initial contact source
- Content engagement level
- Webinar/event attendance
- Downloads of sales collateral
- Website visit frequency
- Social media interaction intensity

**Sales Pipeline Indicators**

**Progression Tracking**

- Current sales pipeline stage
- Days in current stage
- Velocity through previous stages
- Probability of stage conversion
- Historical conversion rates

**Opportunity Indicators**

- Budget signals
- Decision-maker engagement level
- Competitive landscape awareness
- Expressed pain points
- Alignment with solution offerings

**Contextual and Temporal Features**

**Temporal Dynamics**

- Quarter/fiscal period
- Economic indicators
- Seasonal business trends
- Recent company news/events

**Predictive Signals**

- Likelihood of purchase (ML-derived score)
- Potential deal size estimation
- Competitive threat level
- Risk of churn or losing opportunity

Feature Encoding Strategies **bold text**

Numerical Normalization

- Min-Max scaling
- Standard scaling
- Log transformation for skewed distributions

Categorical Encoding

- One-hot encoding
- Ordinal encoding
- Embedding for high-cardinality features

Dimensionality Reduction

- Principal Component Analysis (PCA)
- t-SNE for non-linear dimensionality reduction
- Autoencoders for feature compression

Continuous Learning

- Periodically retrain feature importance weights
- Use feature importance techniques like SHAP values
- Implement online learning for dynamic feature adaptation


In [2]:
def create_state_vector(prospect_data):
    features = []

    # Firmographic encoding
    features.extend([
        normalize_company_size(prospect_data['employee_count']),
        normalize_revenue(prospect_data['annual_revenue']),
        encode_industry(prospect_data['industry']),
        encode_location(prospect_data['headquarters'])
    ])

    # Interaction history
    features.extend([
        count_previous_interactions(prospect_data),
        days_since_last_contact(prospect_data),
        calculate_email_engagement_score(prospect_data),
        get_response_sentiment_score(prospect_data)
    ])

    # Sales pipeline indicators
    features.extend([
        encode_pipeline_stage(prospect_data['current_stage']),
        calculate_stage_velocity(prospect_data),
        estimate_conversion_probability(prospect_data)
    ])

    return np.array(features)

Multi-Dimensional Reward Function for Prospectoro
**bold text**
**Primary Reward Components**

**Deal Closure Rewards**

- Base reward for deal closure
- Scaled reward based on deal value
- Bonus for deals above target

**Pipeline Progression Rewards**

- Incremental rewards for moving through sales stages
- Higher rewards for later-stage progressions
- Penalty for stage regressions

**Secondary Reward Signals**

**Interaction Quality**
Positive reward for meaningful interactions
Higher rewards for:

- Scheduled meetings
- Responded emails
- Successful discovery calls

Negative rewards for:

- Ignored communications
- Abrupt conversation terminations

**Long-Term Value Metrics**

Rewards for customer lifetime potential
Consideration of:

- Potential upsell opportunities
- Referral likelihood
- Strategic fit beyond immediate deal

**Penalty Mechanisms**

Resource Efficiency

Penalties for:

- Excessive time spent on low-probability prospects
- Inefficient communication strategies
- High-touch approaches with low conversion rates

**Opportunity Cost**

Negative rewards for:

- Missed high-potential opportunities
- Prolonged sales cycles
- Misaligned prospect targeting

In [1]:
def calculate_reward(prospect_data, action, outcome):
    base_reward = 0

    # Deal closure reward
    if outcome == 'deal_closed':
        base_reward += calculate_deal_value_score(prospect_data)
        base_reward *= 1 + get_strategic_multiplier(prospect_data)

    # Pipeline progression reward
    stage_progression_score = calculate_stage_progression(prospect_data)
    base_reward += stage_progression_score

    # Interaction quality adjustment
    interaction_quality = assess_interaction_quality(prospect_data, action)
    base_reward *= (1 + interaction_quality)

    # Resource efficiency penalty
    efficiency_penalty = calculate_effort_efficiency(prospect_data)
    base_reward -= efficiency_penalty

    return base_reward

def get_strategic_multiplier(prospect_data):
    """Calculate additional multiplier based on strategic value"""
    multipliers = {
        'high_potential_industry': 0.3,
        'enterprise_segment': 0.2,
        'repeat_customer_potential': 0.1
    }

    total_multiplier = sum(
        multipliers.get(factor, 0)
        for factor in prospect_data.get('strategic_factors', [])
    )

    return min(total_multiplier, 0.5)  # Cap at 50% bonus

**Refinement Strategies**

**Reward Shaping**

- Gradually adjust reward function
- Use domain expert feedback
- Incorporate external performance metrics

**Multi-Objective Optimization**

**Balance between:**

- Short-term revenue
- Long-term customer value
- Sales team efficiency

**Ethical Considerations**

Avoid rewards that might encourage:

- Aggressive sales tactics
- Misrepresentation
- Prioritizing quantity over quality

In [3]:
import numpy as np
import tensorflow as tf
from dataclasses import dataclass, field
from typing import List, Dict, Any

@dataclass
class LeadFeatureExtractor:
    """Specialized feature extraction for B2B lead generation"""

    def extract_features(self, lead_data: Dict[str, Any]) -> np.ndarray:
        """
        Extract a comprehensive feature vector for lead generation

        Key Feature Categories:
        1. Firmographic Data
        2. Digital Engagement Signals
        3. Intent & Readiness Indicators
        4. Historical Interaction Metrics
        """
        features = []

        # Firmographic Features
        features.extend([
            # Numerical Encodings
            self._normalize_employee_count(lead_data.get('employee_count', 0)),
            self._normalize_revenue(lead_data.get('annual_revenue', 0)),

            # Categorical Encodings
            self._encode_industry(lead_data.get('industry', 'unknown')),
            self._encode_company_size_category(lead_data.get('employee_count', 0)),
            self._encode_geographic_region(lead_data.get('region', 'unknown'))
        ])

        # Digital Engagement Features
        features.extend([
            self._calculate_website_interaction_score(lead_data),
            self._calculate_content_engagement_score(lead_data),
            self._get_social_media_activity_level(lead_data)
        ])

        # Intent & Readiness Indicators
        features.extend([
            self._calculate_ideal_customer_profile_match(lead_data),
            self._estimate_purchase_intent(lead_data),
            self._calculate_technology_fit_score(lead_data)
        ])

        # Historical Interaction Metrics
        features.extend([
            self._count_previous_touchpoints(lead_data),
            self._calculate_response_probability(lead_data),
            self._estimate_time_since_last_interaction(lead_data)
        ])

        return np.array(features)

    def _normalize_employee_count(self, count: int) -> float:
        """Normalize employee count using log transformation"""
        return np.log1p(count) / 10.0

    def _normalize_revenue(self, revenue: float) -> float:
        """Normalize annual revenue"""
        return np.log1p(revenue) / 15.0

    def _encode_industry(self, industry: str) -> float:
        """One-hot encoding for industries"""
        industries = ['tech', 'finance', 'healthcare', 'manufacturing', 'other']
        return industries.index(industry.lower()) / len(industries)

    def _encode_company_size_category(self, count: int) -> float:
        """Categorize and encode company size"""
        if count < 50: return 0.2
        elif count < 250: return 0.4
        elif count < 1000: return 0.6
        elif count < 5000: return 0.8
        else: return 1.0

    def _encode_geographic_region(self, region: str) -> float:
        """Encode geographic regions based on market potential"""
        region_scores = {
            'north_america': 1.0,
            'europe': 0.8,
            'asia_pacific': 0.7,
            'latin_america': 0.5,
            'other': 0.3
        }
        return region_scores.get(region.lower(), 0.3)

    def _calculate_website_interaction_score(self, lead_data: Dict) -> float:
        """Score based on website interactions"""
        page_views = lead_data.get('website_page_views', 0)
        time_on_site = lead_data.get('time_on_site', 0)
        return min(np.log1p(page_views * time_on_site) / 10, 1.0)

    def _calculate_content_engagement_score(self, lead_data: Dict) -> float:
        """Score based on content downloads, webinar attendance"""
        downloads = lead_data.get('content_downloads', 0)
        webinar_attendance = lead_data.get('webinar_attendance', False)
        return min((downloads * 0.5) + (1.0 if webinar_attendance else 0), 1.0)

    def _get_social_media_activity_level(self, lead_data: Dict) -> float:
        """Assess social media engagement"""
        linkedin_connections = lead_data.get('linkedin_connections', 0)
        return min(np.log1p(linkedin_connections) / 10, 1.0)

    def _calculate_ideal_customer_profile_match(self, lead_data: Dict) -> float:
        """Calculate how closely lead matches ideal customer profile"""
        match_criteria = [
            lead_data.get('matches_target_industry', False),
            lead_data.get('decision_maker_role', False),
            lead_data.get('budget_range_match', False)
        ]
        return sum(match_criteria) / len(match_criteria)

    def _estimate_purchase_intent(self, lead_data: Dict) -> float:
        """Estimate likelihood of purchase based on behavioral signals"""
        intent_signals = [
            lead_data.get('pricing_page_visit', False),
            lead_data.get('demo_request', False),
            lead_data.get('recent_solution_search', False)
        ]
        return sum(intent_signals) / len(intent_signals)

    def _calculate_technology_fit_score(self, lead_data: Dict) -> float:
        """Assess technological compatibility"""
        current_tech = lead_data.get('current_technology_stack', [])
        compatibility_score = len(set(current_tech) & set(['cloud', 'saas', 'enterprise']))
        return compatibility_score / 3.0

    def _count_previous_touchpoints(self, lead_data: Dict) -> float:
        """Count and normalize previous interactions"""
        touchpoints = [
            lead_data.get('email_interactions', 0),
            lead_data.get('phone_interactions', 0),
            lead_data.get('event_interactions', 0)
        ]
        return min(sum(touchpoints) / 10, 1.0)

    def _calculate_response_probability(self, lead_data: Dict) -> float:
        """Historical response rate"""
        previous_responses = lead_data.get('previous_response_rate', 0)
        return previous_responses

    def _estimate_time_since_last_interaction(self, lead_data: Dict) -> float:
        """Normalize time since last interaction"""
        days_since_last = lead_data.get('days_since_last_interaction', 365)
        return 1 - min(days_since_last / 365, 1.0)

class LeadGenerationActorCritic:
    def __init__(self, state_dim, action_dim):
        self.feature_extractor = LeadFeatureExtractor()

        # Actions could include:
        # 0: Cold Email
        # 1: LinkedIn Connection
        # 2: Targeted Ad Campaign
        # 3: Personalized Content Outreach
        # 4: Direct Phone Call

        self.state_dim = state_dim
        self.action_dim = action_dim

        # Initialize actor and critic networks
        self.actor = self._build_actor_network()
        self.critic = self._build_critic_network()

    def _build_actor_network(self):
        """Create neural network for action selection"""
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(self.state_dim,)),
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(self.action_dim, activation='softmax')
        ])
        model.compile(optimizer='adam')
        return model

    def _build_critic_network(self):
        """Create neural network for value estimation"""
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(self.state_dim,)),
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(1, activation='linear')
        ])
        model.compile(optimizer='adam', loss='mse')
        return model

    def get_action(self, state):
        """Recommend lead generation action"""
        action_probs = self.actor.predict(state.reshape(1, -1))[0]
        action = np.random.choice(self.action_dim, p=action_probs)
        return action, action_probs

    def calculate_lead_generation_reward(self, lead_data, action, outcome):
        """
        Multidimensional reward function for lead generation

        Reward components:
        1. Initial engagement reward
        2. Progression to next stage
        3. Quality of lead interaction
        4. Resource efficiency
        """
        base_reward = 0

        # Engagement Rewards
        if outcome == 'response_received':
            base_reward += 1.0
        elif outcome == 'meeting_scheduled':
            base_reward += 2.0

        # Lead Quality Multiplier
        quality_multipliers = {
            'high_potential': 1.5,
            'medium_potential': 1.0,
            'low_potential': 0.5
        }
        lead_quality = lead_data.get('lead_quality', 'low_potential')
        base_reward *= quality_multipliers.get(lead_quality, 1.0)

        # Action Efficiency Penalty
        action_cost = {
            0: 0.1,   # Cold Email
            1: 0.2,   # LinkedIn Connection
            2: 0.5,   # Targeted Ad Campaign
            3: 0.3,   # Personalized Content
            4: 0.4    # Direct Phone Call
        }
        base_reward -= action_cost.get(action, 0.2)

        return base_reward

def train_lead_generation_model(leads_data):
    """
    Training loop for lead generation Actor-Critic model

    Args:
        leads_data: List of lead dictionaries with interaction history
    """
    feature_extractor = LeadFeatureExtractor()
    state_dim = 15  # Number of extracted features
    action_dim = 5  # Number of lead generation actions

    model = LeadGenerationActorCritic(state_dim, action_dim)

    for lead in leads_data:
        # Extract state features
        state = feature_extractor.extract_features(lead)

        # Recommend action
        action, action_probs = model.get_action(state)

        # Simulate outcome (in real system, this would be actual CRM data)
        outcome = simulate_lead_interaction(lead, action)

        # Calculate reward
        reward = model.calculate_lead_generation_reward(lead, action, outcome)

        # Training logic would go here
        # Update actor and critic networks based on reward

    return model

def simulate_lead_interaction(lead, action):
    """
    Simulate lead interaction outcome
    (Replace with actual CRM interaction tracking)
    """
    outcomes = ['no_response', 'response_received', 'meeting_scheduled']
    return np.random.choice(outcomes, p=[0.6, 0.3, 0.1])

**Prospectoro Lead Gen AI Framework Evaluation**

**Performance Tracking Dimensions:**

**Quantitative Metrics**

**Lead Conversion Rates**

- Overall conversion percentage
- Conversion rates by action type
- Conversion rates by lead segment
- Time-to-conversion analysis

**Economic Impact Metrics**

- Customer Acquisition Cost (CAC)
- Lifetime Value (LTV) of generated leads
- Revenue attribution to AI recommendations
- Return on Lead Generation Investment

**Qualitative Assessment**

**Sales Team Feedback Mechanism**

- Recommendation accuracy rating
- Usefulness of AI-suggested actions
- Alignment with sales team intuition
- Ease of integration into workflow

In [4]:
class ProspectoroModelMonitor:
    def __init__(self):
        self.performance_metrics = {
            'conversion_rates': {},
            'action_effectiveness': {},
            'lead_quality_scores': {},
            'model_confidence_levels': {}
        }
        self.drift_detection = DriftDetector()
        self.bias_analyzer = BiasAnalyzer()

    def track_lead_generation_performance(self, leads_data):
        """
        Comprehensive performance tracking across multiple dimensions
        """
        # Conversion rate analysis
        conversion_rates = self._calculate_conversion_rates(leads_data)

        # Action effectiveness scoring
        action_effectiveness = self._analyze_action_effectiveness(leads_data)

        # Lead quality assessment
        lead_quality_scores = self._evaluate_lead_quality(leads_data)

        # Model confidence estimation
        model_confidence = self._estimate_model_confidence(leads_data)

        return {
            'conversion_rates': conversion_rates,
            'action_effectiveness': action_effectiveness,
            'lead_quality_scores': lead_quality_scores,
            'model_confidence': model_confidence
        }

    def detect_model_drift(self, historical_data, current_data):
        """
        Identify potential concept drift or data distribution changes
        """
        drift_indicators = self.drift_detection.analyze_distribution_shift(
            historical_data,
            current_data
        )
        return drift_indicators

    def analyze_model_bias(self, leads_data):
        """
        Comprehensive bias detection across multiple dimensions
        """
        bias_report = self.bias_analyzer.generate_bias_report(leads_data)
        return bias_report

class DriftDetector:
    def analyze_distribution_shift(self, historical_data, current_data):
        """
        Detect statistical differences between historical and current data
        """
        drift_metrics = {
            'feature_distribution_changes': self._calculate_feature_drift(historical_data, current_data),
            'performance_consistency': self._measure_performance_stability(historical_data, current_data),
            'segment_performance_variance': self._analyze_segment_performance_drift(historical_data, current_data)
        }
        return drift_metrics

class BiasAnalyzer:
    def generate_bias_report(self, leads_data):
        """
        Comprehensive bias analysis across multiple dimensions
        """
        return {
            'demographic_representation': self._check_demographic_fairness(leads_data),
            'industry_bias': self._analyze_industry_representation(leads_data),
            'geographic_bias': self._check_geographic_coverage(leads_data),
            'socioeconomic_bias': self._evaluate_company_size_representation(leads_data)
        }

**Interpretability Techniques:**
- SHAP value analysis
- Feature importance tracking
- Decision boundary visualisation

**Anomaly Detection:**
- Identify unusual lead generation patterns
- Flag potential model degradation
- Detect exceptional performance scenarios

**Continuous Improvement Workflow**

**Regular Model Audits**

- Monthly performance reviews
- Quarterly comprehensive evaluations
- Annual deep-dive analysis

**Feedback Integration **

- Sales team input mechanism
- Continuous model retraining
- Adaptive learning framework

**Data Collection Infrastructure**

- Centralized data warehouse
- Real-time performance tracking
- Secure, compliant data management

**Technological Requirements**

- Low-latency monitoring system
- Scalable cloud infrastructure
- Robust data pipeline
- Real-time alerting mechanisms

**Machine Learning Observability**

- Model version tracking
- Performance degradation alerts
- Automated retraining triggers



In [1]:
pip install shap

SHAP Value Analysis:

### Key Insights and Recommendations

1. **SHAP Value Analysis Benefits**
   - Provides model interpretability
   - Identifies most influential lead generation features
   - Helps understand model decision-making process
   - Supports continuous model improvement

2. **Data Infrastructure Considerations**
   - Prioritize data quality and consistency
   - Build flexible, scalable architecture
   - Implement robust security measures
   - Enable real-time feature generation

### Practical Next Steps

1. Conduct initial data source audit
2. Design proof-of-concept data pipeline
3. Implement initial SHAP analysis framework
4. Create prototype feature store
5. Develop monitoring dashboards

Would you like me to elaborate on any specific aspect of the SHAP analysis or data collection infrastructure?

In [2]:
import shap
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

class ProspectороSHAPAnalyzer:
    def __init__(self, model, feature_names):
        """
        Initialize SHAP analysis for lead generation model

        Args:
            model: Trained machine learning model
            feature_names: List of feature names for interpretation
        """
        self.model = model
        self.feature_names = feature_names

    def compute_shap_values(self, X_sample):
        """
        Compute SHAP values for model interpretability

        Args:
            X_sample: Sample of lead features for analysis

        Returns:
            Comprehensive SHAP analysis report
        """
        # Use KernelExplainer for complex models
        explainer = shap.KernelExplainer(self.model.predict, X_sample)
        shap_values = explainer.shap_values(X_sample)

        # Generate detailed analysis
        return self._analyze_shap_results(shap_values, X_sample)

    def _analyze_shap_results(self, shap_values, X_sample):
        """
        Comprehensive analysis of SHAP values

        Provides insights into:
        1. Feature importance
        2. Feature impact direction
        3. Interaction effects
        """
        analysis = {
            'global_feature_importance': self._calculate_global_importance(shap_values),
            'feature_impact_summary': self._summarize_feature_impacts(shap_values, X_sample),
            'top_positive_features': self._get_top_positive_features(shap_values),
            'top_negative_features': self._get_top_negative_features(shap_values)
        }

        return analysis

    def _calculate_global_importance(self, shap_values):
        """Calculate overall feature importance across all samples"""
        return np.abs(shap_values).mean(axis=0)

    def _summarize_feature_impacts(self, shap_values, X_sample):
        """
        Create a detailed summary of how features impact model predictions

        Returns a DataFrame with:
        - Mean SHAP value
        - Variance of SHAP value
        - Direction of impact
        """
        feature_impacts = pd.DataFrame({
            'feature': self.feature_names,
            'mean_shap_value': np.abs(shap_values).mean(axis=0),
            'impact_direction': np.sign(shap_values).mean(axis=0)
        })

        return feature_impacts.sort_values('mean_shap_value', ascending=False)

    def _get_top_positive_features(self, shap_values, top_k=5):
        """Identify features with strongest positive impact"""
        feature_importances = np.abs(shap_values).mean(axis=0)
        positive_mask = shap_values.mean(axis=0) > 0
        top_positive = np.argsort(feature_importances * positive_mask)[-top_k:]

        return [self.feature_names[i] for i in top_positive]

    def _get_top_negative_features(self, shap_values, top_k=5):
        """Identify features with strongest negative impact"""
        feature_importances = np.abs(shap_values).mean(axis=0)
        negative_mask = shap_values.mean(axis=0) < 0
        top_negative = np.argsort(feature_importances * negative_mask)[-top_k:]

        return [self.feature_names[i] for i in top_negative]

    def generate_visualization(self, shap_values, X_sample):
        """
        Create multiple SHAP visualizations

        Types of visualizations:
        1. Summary plot
        2. Dependence plots
        3. Force plots for individual predictions
        """
        import matplotlib.pyplot as plt

        # Summary plot
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_sample, feature_names=self.feature_names)
        plt.title('Feature Importance and Impact Direction')
        plt.tight_layout()
        plt.savefig('shap_summary_plot.png')
        plt.close()

        # Dependence plots for top features
        top_features = self._get_top_positive_features(shap_values) + \
                       self._get_top_negative_features(shap_values)

        for feature in top_features:
            plt.figure(figsize=(10, 6))
            feature_index = self.feature_names.index(feature)
            shap.dependence_plot(
                feature_index,
                shap_values,
                X_sample,
                feature_names=self.feature_names
            )
            plt.title(f'SHAP Dependence Plot for {feature}')
            plt.tight_layout()
            plt.savefig(f'shap_dependence_{feature}.png')
            plt.close()

def example_shap_analysis(leads_data):
    """
    Example workflow for SHAP analysis in lead generation
    """
    # Prepare data
    X = leads_data[feature_columns]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Assume pre-trained model
    model = load_trained_lead_generation_model()

    # Initialize SHAP analyzer
    feature_names = feature_columns
    shap_analyzer = ProspectороSHAPAnalyzer(model, feature_names)

    # Sample subset for SHAP analysis
    sample_indices = np.random.choice(len(X_scaled), 100, replace=False)
    X_sample = X_scaled[sample_indices]

    # Compute and visualize SHAP values
    shap_analysis = shap_analyzer.compute_shap_values(X_sample)
    shap_analyzer.generate_visualization(shap_analysis['shap_values'], X_sample)

    return shap_analysis

**Multi Agent Reinforcement Learning (MARL):**

- Create an ensemble of specialised AI agents, each focusing on different aspects of lead generation.

**Agent Specialisations:**

**Targeting Agent:**
- identifies most promising lead segments
- learns optimal targeting strategies
- focuses on intial lead qualification

**Engagement Agent:**

- determines best communication channels
- optimises messaging and outreach timing
- learns personalisation strategies

**Conversion Agent:**

- tracks lead progression through sales funnel
- identifies conversion acceleration tecnniques

**Resource Allocation Agent:**

- optmises sales team effort
- balances high-potential and long-tail leads
- manages time and resources investments






**Implementation Approach:**


In [3]:
class ContextualBanditLeadGen:
    def __init__(self, n_actions, context_dim):
        """
        Initialize contextual bandit for lead generation

        Args:
        - n_actions: Number of possible lead generation actions
        - context_dim: Dimensionality of lead context features
        """
        self.n_actions = n_actions
        self.context_dim = context_dim

        # Bayesian linear regression for each action
        self.models = [
            BayesianLinearRegression(context_dim)
            for _ in range(n_actions)
        ]

    def select_action(self, context):
        """
        Select action using Thompson sampling

        Considers:
        - Uncertainty in action value
        - Potential reward distribution
        """
        action_values = []
        for model in self.models:
            # Sample from posterior distribution
            value_sample = model.sample_posterior(context)
            action_values.append(value_sample)

        return np.argmax(action_values)

    def update(self, context, action, reward):
        """
        Update model based on observed outcome

        Incorporates:
        - Observed reward
        - Context features
        - Bayesian parameter updates
        """
        self.models[action].update(context, reward)

**Advanced Exploration Strategy:**

- Probabilistic approach to action selection
- Adaptive learning across lead segments
- Bayesian updating of action probabilities
- Prior belief distribution of each action

In [4]:
class ContextualBanditLeadGen:
    def __init__(self, n_actions, context_dim):
        """
        Initialize contextual bandit for lead generation

        Args:
        - n_actions: Number of possible lead generation actions
        - context_dim: Dimensionality of lead context features
        """
        self.n_actions = n_actions
        self.context_dim = context_dim

        # Bayesian linear regression for each action
        self.models = [
            BayesianLinearRegression(context_dim)
            for _ in range(n_actions)
        ]

    def select_action(self, context):
        """
        Select action using Thompson sampling

        Considers:
        - Uncertainty in action value
        - Potential reward distribution
        """
        action_values = []
        for model in self.models:
            # Sample from posterior distribution
            value_sample = model.sample_posterior(context)
            action_values.append(value_sample)

        return np.argmax(action_values)

    def update(self, context, action, reward):
        """
        Update model based on observed outcome

        Incorporates:
        - Observed reward
        - Context features
        - Bayesian parameter updates
        """
        self.models[action].update(context, reward)